# 03 - Truth Validation

Goal: validate the analysis-facing objects with matching, confusion matrices, and a simple vertex-resolution diagnostic.

This is the compressed version of the PID and primary/vertex validation notebooks from the full workshop.

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from spine.driver import Driver

# Tutorial data convention at FNAL/EAF:
#   LARCV_DATA_DIR/DETECTOR/DETECTOR_TAG.root
#   HDF5_DATA_DIR/DETECTOR/DETECTOR_TAG_spine.hdf5
LARCV_DATA_DIR = Path("/exp/dune/data/users/drielsma/npc-ddas/larcv")
HDF5_DATA_DIR = Path("/exp/dune/data/users/drielsma/npc-ddas/reco")

# Edit these two lines to switch samples.
# Common detector examples: "icarus", "sbnd", "2x2", "nd-lar",
# "protodune-sp", "protodune-vd"
DETECTOR = "generic"
TAG = "tutorial"
GEOMETRY = None if DETECTOR == "generic" else DETECTOR

LARCV_PATH = LARCV_DATA_DIR / DETECTOR / f"{DETECTOR}_{TAG}.root"
DATA_PATH = HDF5_DATA_DIR / DETECTOR / f"{DETECTOR}_{TAG}_spine.hdf5"

CONFIG_CANDIDATES = [
    Path("../config/read_spine_hdf5.yaml"),
    Path("tutorials/config/read_spine_hdf5.yaml"),
]
CONFIG_PATH = next((path for path in CONFIG_CANDIDATES if path.exists()), None)

DATA_PATH = str(DATA_PATH)
if CONFIG_PATH is None:
    raise RuntimeError("Could not find tutorials/config/read_spine_hdf5.yaml")

cfg_text = CONFIG_PATH.read_text().replace("DATA_PATH", DATA_PATH)
cfg = yaml.safe_load(cfg_text)
if GEOMETRY:
    cfg["geo"] = {"detector": GEOMETRY}
driver = Driver(cfg)
print(f"Opened {DATA_PATH}")
print(f"Entries: {len(driver)}")
if GEOMETRY:
    print(f"Detector geometry: {GEOMETRY}")
print(f"Companion LArCV path: {LARCV_PATH}")


In [ ]:
from sklearn.metrics import confusion_matrix

N_ENTRIES = min(len(driver), 100)
MATCH_THRESHOLD = 0.1
print(f"Validating {N_ENTRIES} entries with overlap threshold {MATCH_THRESHOLD}")

In [ ]:
particle_rows = []
primary_rows = []
vertex_rows = []

for entry in range(N_ENTRIES):
    data = driver.process(entry=entry)

    pairs = data.get("particle_matches_t2r", [])
    overlaps = data.get("particle_matches_t2r_overlap", np.ones(len(pairs)))
    for i, (truth_p, reco_p) in enumerate(pairs):
        overlap = overlaps[i]
        if overlap < MATCH_THRESHOLD:
            continue
        if getattr(truth_p, "pid", -1) < 0 or getattr(reco_p, "pid", -1) < 0:
            continue
        particle_rows.append({
            "entry": entry,
            "overlap": overlap,
            "true_pid": truth_p.pid,
            "reco_pid": reco_p.pid,
            "true_primary": bool(getattr(truth_p, "is_primary", False)),
            "reco_primary": bool(getattr(reco_p, "is_primary", False)),
            "true_size": getattr(truth_p, "size", np.nan),
            "reco_size": getattr(reco_p, "size", np.nan),
            "true_nu_id": getattr(truth_p, "nu_id", -1),
        })
        primary_rows.append({
            "entry": entry,
            "truth": bool(getattr(truth_p, "is_primary", False)),
            "reco": bool(getattr(reco_p, "is_primary", False)),
            "overlap": overlap,
            "true_nu_id": getattr(truth_p, "nu_id", -1),
        })

    ia_pairs = data.get("interaction_matches_t2r", [])
    ia_overlaps = data.get("interaction_matches_t2r_overlap", np.ones(len(ia_pairs)))
    for i, (truth_ia, reco_ia) in enumerate(ia_pairs):
        overlap = ia_overlaps[i]
        if overlap < MATCH_THRESHOLD:
            continue
        if not hasattr(truth_ia, "vertex") or not hasattr(reco_ia, "vertex"):
            continue
        vertex_rows.append({
            "entry": entry,
            "truth_interaction_id": getattr(truth_ia, "id", -1),
            "reco_interaction_id": getattr(reco_ia, "id", -1),
            "overlap": overlap,
            "true_nu_id": getattr(truth_ia, "nu_id", -1),
            "vertex_distance_cm": float(np.linalg.norm(truth_ia.vertex - reco_ia.vertex)),
        })

particles = pd.DataFrame(particle_rows)
primaries = pd.DataFrame(primary_rows)
vertices = pd.DataFrame(vertex_rows)

display(particles.head())
display(vertices.head())

In [ ]:
PID_LABELS = ["photon", "electron", "muon", "pion", "proton"]
valid_pid = particles.query("0 <= true_pid <= 4 and 0 <= reco_pid <= 4")

cm_counts = confusion_matrix(valid_pid["true_pid"], valid_pid["reco_pid"], labels=[0, 1, 2, 3, 4])
cm_norm = confusion_matrix(valid_pid["true_pid"], valid_pid["reco_pid"], labels=[0, 1, 2, 3, 4], normalize="true")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap="Blues")
plt.colorbar(im, ax=ax, label="row-normalized fraction")
ax.set_xticks(range(5), PID_LABELS, rotation=45, ha="right")
ax.set_yticks(range(5), PID_LABELS)
ax.set_xlabel("reco PID")
ax.set_ylabel("true PID")
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}\n({cm_counts[i, j]})", ha="center", va="center", fontsize=8)
plt.tight_layout()

In [ ]:
mpv_primary = primaries[primaries["true_nu_id"] > -1]
primary_counts = confusion_matrix(mpv_primary["truth"], mpv_primary["reco"], labels=[False, True])
primary_norm = confusion_matrix(mpv_primary["truth"], mpv_primary["reco"], labels=[False, True], normalize="true")

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(primary_norm, vmin=0, vmax=1, cmap="Greens")
plt.colorbar(im, ax=ax, label="row-normalized fraction")
ax.set_xticks([0, 1], ["non-primary", "primary"], rotation=30, ha="right")
ax.set_yticks([0, 1], ["non-primary", "primary"])
ax.set_xlabel("reco")
ax.set_ylabel("truth")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{primary_norm[i, j]:.2f}\n({primary_counts[i, j]})", ha="center", va="center")
plt.tight_layout()

In [ ]:
mpv_vertices = vertices[vertices["true_nu_id"] > -1]
fig, ax = plt.subplots(figsize=(6, 3))
mpv_vertices["vertex_distance_cm"].hist(ax=ax, bins=30)
ax.set_xlabel("truth-reco vertex distance [cm]")
ax.set_ylabel("matched interactions")
ax.grid(alpha=0.3)

bad_vertices = mpv_vertices.sort_values("vertex_distance_cm", ascending=False).head(10)
bad_vertices

## Live exercise

Pick one bad vertex or one PID confusion mode and send that `(entry, interaction_id)` back to Spinal Tap. Decide whether the problem is segmentation, fragmentation, interaction clustering, PID, primary ID, or vertexing.

## Offline extensions

- Split PID confusion by particle length, deposited charge, containment, or detector module boundary.
- Build efficiency and purity curves for the selection in Notebook 2.
- Compare validation metrics before and after changing a production modifier or post-processing threshold.
- Turn one recurring failure mode into a short debugging note with event IDs and screenshots.

## Real-analysis checklist

- Record the exact SPINE and `spine-prod` versions used to make the file.
- Keep the inference config, weight path or tag, detector geometry, and input file provenance.
- Validate object definitions before optimizing cuts.
- Keep event-display debugging tied to table rows; screenshots without entry IDs are not reproducible.